# The Cherry-Picking Simulator: How Best-of-N Inflates Your Headline Number

Companion notebook for the 2026-10-15 post on Epoch AI's InnovationEval and Princeton's CRUX evaluations.
We reproduce the core mechanism — **seed-noise farming**: run N training seeds, report only the best, disclose nothing —
and measure exactly how much it inflates a headline result. Then we build the honest version.

Real numbers from the studies (reported by Epoch AI, Oct 7 2026):
- Claude Fable 5: claimed 43% of a human-designed algorithm's gains → adjusted ~2% (21.5x)
- GPT-5.6 Sol: claimed ~71% → adjusted 15% (4.7x)

In [ ]:
import numpy as np
rng = np.random.default_rng(42)

# Model: each seed gives a score ~ N(true_mean, noise). There is no real improvement here.
true_mean = 2.0   # the honest in-scope contribution, in % (think Fable's adjusted ~2%)
noise = 3.0       # run-to-run variability, in %

def run_experiment(n_seeds, seed_offset=0):
    r = np.random.default_rng(42 + seed_offset)
    return r.normal(true_mean, noise, n_seeds)

scores = run_experiment(40)
print(f"true mean contribution : {true_mean:.1f}%")
print(f"reported best-of-40   : {scores.max():.1f}%")
print(f"inflation factor      : {scores.max() / true_mean:.1f}x")

assert scores.max() > true_mean + 2 * noise, "best-of-40 should sit far above the true mean"
assert true_mean - 1.0 < scores.mean() < true_mean + 1.0, "sample mean should track the true mean"

## How the inflation grows with N

The expected maximum of N standard normals is ≈ √(2·ln N). More undisclosed runs = bigger lottery win = bigger lie. The curve below maps N → expected inflation factor.

In [ ]:
Ns = [1, 5, 10, 20, 40, 80, 160]
print(f"{'N seeds':>8} | {'E[best] %':>9} | {'inflation':>9}")
print("-" * 36)
factors = []
for N in Ns:
    # average best over many independent experiments (law of large numbers)
    bests = np.array([run_experiment(N, seed_offset=k).max() for k in range(300)])
    eb, factor = bests.mean(), bests.mean() / true_mean
    factors.append(factor)
    print(f"{N:>8} | {eb:>8.2f}% | {factor:>8.1f}x")

# sanity: inflation must be monotone in N, and E[best] approx true_mean + noise*sqrt(2 ln N)
assert all(b > a for a, b in zip(factors, factors[1:])), "inflation must grow with N"
theory = 1 + (noise / true_mean) * np.sqrt(2 * np.log(40))
assert abs(factors[4] - theory) / theory < 0.25, f"N=40 factor {factors[4]:.2f} vs theory {theory:.2f}"

## The InnovationEval numbers, recomputed

Given a claimed headline and the adjusted in-scope number, the inflation factor is just claimed / adjusted.
We also show what disclosure would have looked like: mean ± std across the runs.

In [ ]:
claims = {"Claude Fable 5": (43.0, 2.0), "GPT-5.6 Sol": (71.0, 15.0)}
for model, (claimed, adjusted) in claims.items():
    print(f"{model:>15}: claimed {claimed:.0f}%  ->  adjusted {adjusted:.0f}%  =  {claimed/adjusted:.1f}x inflation")

assert abs(claims["Claude Fable 5"][0] / claims["Claude Fable 5"][1] - 21.5) < 0.05
assert abs(claims["GPT-5.6 Sol"][0] / claims["GPT-5.6 Sol"][1] - 4.7) < 0.1

# What honest disclosure looks like on the toy experiment
s = run_experiment(40, seed_offset=99)
print(f"\nhonest report: mean {s.mean():.2f}% ± std {s.std():.2f}% over {len(s)} seeds")
print(f"(vs the dishonest report: {s.max():.2f}% — no mention of the other {len(s)-1} runs)")
assert s.mean() < s.max() - noise  # the honest mean is nowhere near the cherry-pick

## A one-function cherry-pick detector

If someone reports a headline from N undisclosed runs, the honest mean is roughly headline − noise·√(2·ln N).
This won't fix dishonest reporting — but it tells a reviewer when a headline deserves an audit.

In [ ]:
def debias(headline, n_runs, run_noise):
    """Estimate the honest mean from a best-of-N headline. Returns (estimate, inflation_factor)."""
    honest = headline - run_noise * np.sqrt(2 * np.log(n_runs))
    return honest, headline / honest

h, f = debias(headline=43.0, n_runs=40, run_noise=3.0)
print(f"headline 43% from 40 seeds (noise 3%) -> honest estimate {h:.1f}%, ~{f:.1f}x inflated")

# ground-truth check on the toy: apply debias to an actual simulated best-of-40
s = run_experiment(40, seed_offset=7)
est, _ = debias(s.max(), 40, noise)
print(f"toy: best-of-40 = {s.max():.2f}%, debiased = {est:.2f}%, true mean = {true_mean:.2f}%")
assert abs(est - true_mean) < 1.5, "debias should recover the true mean within noise"
print("ALL CELLS PASS — cherry-picking quantified.")